# Checking out India's Unemployment Data (2019-2020)

I wanted to see how bad the job market actually got during the initial Covid lockdowns. I grabbed two datasets from Kaggle spanning 2019 to 2020 to see if the numbers match the chaos we saw on the news.

I'm going to load both files, merge them together so we have a full timeline from mid-2019 all the way to late 2020, clean them up a bit, and plot a few things to get a feel for the trend.

In [ ]:
# pulling in the usual libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# hiding those annoying pink warning boxes pandas throws sometimes
warnings.filterwarnings('ignore')

# setting up a clean look for the charts later
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## Reading and merging the data

Going to load the two CSVs. Sometimes these files have weird extra spaces in the headers or overlapping records, so I'll strip those out and combine them so we only count every Region/Date once.

In [ ]:
dataset_older = pd.read_csv('../data/Unemployment in India.csv')
dataset_newer = pd.read_csv('../data/Unemployment_Rate_upto_11_2020.csv')

# cleaning up the column names because spaces are annoying
dataset_older.columns = dataset_older.columns.str.strip()
dataset_newer.columns = dataset_newer.columns.str.strip()

# dropping empty rows just in case
dataset_older = dataset_older.dropna()
dataset_newer = dataset_newer.dropna()

# fixing dates so pandas knows it's actually a timeline
dataset_older['Date'] = pd.to_datetime(dataset_older['Date'].str.strip(), format='%d-%m-%Y', errors='coerce')
dataset_newer['Date'] = pd.to_datetime(dataset_newer['Date'].str.strip(), format='%d-%m-%Y', errors='coerce')

# To combine them nicely, I'll drop the extra Region mapping columns in the newer dataset
if 'Region.1' in dataset_newer.columns:
    dataset_newer = dataset_newer.drop(columns=['Region.1', 'longitude', 'latitude'])

india_labor_snapshot = pd.concat([dataset_older, dataset_newer], ignore_index=True)

# dropping any overlap between the two files
india_labor_snapshot = india_labor_snapshot.drop_duplicates(subset=['Region', 'Date', 'Area'])

# tagging the covid months (basically anything from March 2020 onwards)
india_labor_snapshot['is_covid_era'] = india_labor_snapshot['Date'] >= pd.to_datetime('2020-03-01')

india_labor_snapshot.head()

## How much did it actually jump?

I'm curious what the flat average was before Covid hit versus during the thick of the lockdowns across this combined timeline.

In [ ]:
# looking at the overall average first
print(f"Overall Average Unemployment: {india_labor_snapshot['Estimated Unemployment Rate (%)'].mean():.2f}%")

# splitting it by our covid tag to see the difference
era_split = india_labor_snapshot.groupby('is_covid_era')['Estimated Unemployment Rate (%)'].mean().reset_index()
era_split['Timeframe'] = era_split['is_covid_era'].map({False: 'Pre-Covid', True: 'During-Covid'})

era_split[['Timeframe', 'Estimated Unemployment Rate (%)']]

Using both datasets gives us an even bigger jump! From roughly **9.48%** before Covid all the way up to **15.31%**. That **5.82%** spike means a huge part of the national workforce abruptly lost their jobs. 

Let's map this out to see exactly when it broke.

In [ ]:
plt.figure(figsize=(14, 6))

# crunching down to a single average per month
monthly_unemp_curve = india_labor_snapshot.groupby('Date')['Estimated Unemployment Rate (%)'].mean().reset_index()

sns.lineplot(data=monthly_unemp_curve, x='Date', y='Estimated Unemployment Rate (%)', marker='o', color='crimson')

plt.title('Unemployment Rate Curve (Combined 2019-2020)', fontsize=14)
plt.ylabel('Unemployment Rate (%)')
plt.xlabel('Month')

# drawing a line right when lockdowns started to see the immediate effect
plt.axvline(pd.to_datetime('2020-03-24'), color='black', linestyle='--', alpha=0.6, label='Lockdowns Announced')
plt.legend()
plt.show()

That spike in April and May 2020 is crazy. It went almost vertical right after the lockdown started and took months to start calming down.

I want to see if this was consistent across the country or if some states took the hit worse than others.

In [ ]:
plt.figure(figsize=(16, 7))

# getting the average for each state, split by covid/pre-covid
region_level_summary = india_labor_snapshot.groupby(['Region', 'is_covid_era'])['Estimated Unemployment Rate (%)'].mean().unstack().reset_index()
region_level_summary.columns = ['Region', 'Before', 'During']

# squishing the table back down so seaborn can plot it nicely
plot_ready_regions = pd.melt(region_level_summary, id_vars='Region', var_name='Period', value_name='Rate')
plot_ready_regions = plot_ready_regions.sort_values(by='Rate', ascending=False)

sns.barplot(data=plot_ready_regions, x='Region', y='Rate', hue='Period', palette='viridis')
plt.title('How Hard Did Different States Get Hit? (Combined Data)', fontsize=14)
plt.xticks(rotation=90)
plt.ylabel('Avg Unemployment Rate (%)')
plt.xlabel('')
plt.show()

Looking at this, it definitely wasn't an even spread. 

**Haryana** absolutely tanked, hitting a crazy **31.62%** average unemployment during the covid window. It's wild to think almost a third of the workforce there was unemployed for a stretch. Meanwhile, a few other states barely saw a bump at all compared to their normal baseline.

It's pretty eye-opening to see the raw numbers behind the 2020 economic stall rather than just remembering the news.